## PER2526, grupo de laboratorio L1-3CO11
## A3. Prueba práctica de laboratorio de B2 (cuaderno) (1.25 puntos)
## 26 de mayo de 2026

**Ejercicio:** $\;$ Suponiendo que queremos adaptar el modelo "resnet50.fb_swsl_ig1b_ft_in1k" a la tarea Cifar-10 siguiendo los pasos realizados en las sesiones de prácticas, completa el siguiente cuaderno para preparar un experimento en que se realize un finetuning. En el experimento, entrenaremos el modelo durante 5 épocas utilizando una transformación de reflejo horizontal aleatorio con una probabilidad de `0.7`, además de un scheduler `CosineAnnealing` con un learning rate minimo de `1e-6`.

**Nota:** $\;$ Antes de empezar el experimento, debes introducir las **últimas 3** cifras de tu DNI/NIE en el campo `dni` de la segunda celda del cuaderno

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import timm
from torchvision.transforms import v2
from torch.utils.data import DataLoader
import datasets

In [8]:
dni=515 # Completar

In [9]:
def exp(model, device, train_loader, test_loader, optimizer, scheduler, epochs=15):
    loss_fn = torch.nn.CrossEntropyLoss()
    for epoch in range(epochs):
        print(f'Epoch {epoch}: train:', end=' ')
        model.train(); trsize, trbatches, trloss, tracc = 0, 0, 0, 0
        for batch in train_loader:
            X, y = batch['X'].to(device), batch['y'].to(device); trsize += len(X); trbatches += 1
            pred = model(X); loss = loss_fn(pred, y); trloss += loss.item()
            tracc += (pred.argmax(1) == y).type(torch.float).sum().item()
            loss.backward(); optimizer.step(); optimizer.zero_grad(); scheduler.step()
        trloss /= trbatches; tracc /= trsize
        print(f'loss {trloss:g} acc {tracc:.2%} test:', end=' ')
        model.eval(); tesize, tebatches, teloss, teacc = 0, 0, 0, 0
        with torch.no_grad():
            for batch in test_loader:
                X, y = batch['X'].to(device), batch['y'].to(device); tesize += len(X); tebatches += 1
                pred = model(X); teloss += loss_fn(pred, y).item()
                teacc += (pred.argmax(1) == y).type(torch.float).sum().item()
        teloss /= tebatches; teacc /= tesize
        print(f'loss {teloss:g} acc {teacc:.2%}')

In [10]:
ds = datasets.load_dataset("uoft-cs/cifar10").rename_columns({'img': 'X', 'label': 'y'})
train_ds = ds['train'].to_iterable_dataset(num_shards=1024).shuffle(seed=dni)
test_ds = ds['test'].to_iterable_dataset(num_shards=1024)

In [11]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
model_name = "resnet50.fb_swsl_ig1b_ft_in1k"
model = timm.create_model(model_name, pretrained=True, num_classes=10).to(device) # Completar

In [12]:
data_cfg = timm.data.resolve_data_config(model.pretrained_cfg)
data_cfg['input_size'] = (3, 32, 32) # Completar
timm.data.create_transform(**data_cfg)

Compose(
    Resize(size=36, interpolation=bilinear, max_size=None, antialias=True)
    CenterCrop(size=(32, 32))
    MaybeToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)

In [13]:
train_transform = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.uint8, scale=True),
        v2.RandomHorizontalFlip(p=0.7),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=data_cfg["mean"], std=data_cfg["std"]),
    ]) # Completar
def train_prep(example):
    return { 'X' : train_transform(example['X']), 'y' : example['y']}
train_ds = train_ds.map(train_prep, batched=True)
train_loader = DataLoader(train_ds, batch_size=32)

test_transform = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=data_cfg["mean"], std=data_cfg["std"]),
    ]) # Completar
def test_prep(example):
    return { 'X' : test_transform(example['X']), 'y' : example['y']}
test_ds = test_ds.map(test_prep, batched=True)
test_loader = DataLoader(test_ds, batch_size=32)

In [14]:
for p in model.parameters():
    p.requires_grad = False

for p in model.fc.parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3) 
steps = 5
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, steps, eta_min=1e-6) # Completar

exp(model, device, train_loader, test_loader, optimizer, scheduler, epochs=steps)

Epoch 0: train: loss 1.66808 acc 43.70% test: loss 1.67959 acc 48.79%
Epoch 1: train: loss 1.50434 acc 49.26% test: loss 1.73117 acc 47.68%
Epoch 2: train: loss 1.49743 acc 49.64% test: loss 1.81714 acc 47.78%
Epoch 3: train: loss 1.50786 acc 49.44% test: loss 1.88415 acc 47.31%
Epoch 4: train: loss 1.49008 acc 50.03% test: loss 1.8484 acc 47.17%



<p style="page-break-after:always;"></p>
